# Dimensionality Reduction — Method Comparison
## Machine learning for energy applications — Module 4, Exercise 4

In this exercise we bring together three dimensionality-reduction methods — **PCA**,
**t-SNE**, and **UMAP** — and compare them on the petrophysical dataset under one protocol:
the same rows, the same scaling, fixed seeds and recorded settings for every method.

We evaluate:
1. Visual quality of 2D embeddings
2. Computational time (on this machine, for these 1,000 rows)
3. Neighbourhood preservation (trustworthiness at k = 10) and stability across seeds and settings
4. How separated the known facies look in each map (silhouette score)
5. Trade-offs and practical guidance

> **Exploratory analysis.** The facies labels are kept out of the inputs; they are used only to
> colour and score the maps after fitting. All fits use all 1,000 rows, so the results describe
> this table, not a model for new wells.

---
## 0 — Imports and Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Try importing UMAP (optional package: pip install umap-learn)
# (Without ipywidgets, importing umap may print a harmless TqdmWarning about IProgress.)
try:
    import umap
    UMAP_AVAILABLE = True
    print("umap-learn is available.")
except ImportError:
    UMAP_AVAILABLE = False
    print("WARNING: umap-learn is NOT installed.")
    print("Install with: pip install umap-learn")
    print("UMAP cells will be skipped.")

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Prepare the Data

We use the petrophysical dataset with 10 log channels and a facies label (integer codes 0, 1
and 2; the file gives no facies names). The copy of `petro.csv` in this folder is identical to
the one in `03_umap`.

In [ ]:
df = pd.read_csv("petro.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
feature_cols = [c for c in df.columns if c != "facies"]
X = df[feature_cols].values
y = df["facies"].values          # used only to colour and score the maps after fitting
assert "facies" not in feature_cols

# Exploratory: the scaler is fitted on all rows, and every method gets the same scaled matrix
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_classes = len(np.unique(y))
print(f"Features : {len(feature_cols)}")
print(f"Samples  : {X.shape[0]}")
print(f"Classes  : {n_classes}")

---
## 2 — Apply All Three Methods

We time each method and store the 2D embeddings for comparison. Every method gets the same
scaled matrix, seeds are fixed, and the settings are written in the code. Times are for this
machine and these 1,000 rows only; the first UMAP call also includes one-off compilation, so it
overstates the cost of later runs. Each map is then scored with trustworthiness at k = 10.

In [ ]:
results = {}

# --- PCA ---
t0 = time.time()
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
t_pca = time.time() - t0
results["PCA"] = {"embedding": X_pca, "time": t_pca}
print(f"PCA            : {t_pca:.4f}s  (explained var = {pca.explained_variance_ratio_.sum():.3f})")

# --- t-SNE ---
t0 = time.time()
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)
t_tsne = time.time() - t0
results["t-SNE"] = {"embedding": X_tsne, "time": t_tsne}
print(f"t-SNE          : {t_tsne:.4f}s  (KL div = {tsne.kl_divergence_:.4f})")

# --- UMAP ---
if UMAP_AVAILABLE:
    t0 = time.time()
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        random_state=RANDOM_STATE, n_jobs=1)
    X_umap = reducer.fit_transform(X_scaled)
    t_umap = time.time() - t0
    results["UMAP"] = {"embedding": X_umap, "time": t_umap}
    print(f"UMAP           : {t_umap:.4f}s")
else:
    print("UMAP skipped — not installed.")

# Trustworthiness of every map on the same rows, at the same k (outside the timed blocks)
for method, res in results.items():
    res["trust"] = trustworthiness(X_scaled, res["embedding"], n_neighbors=10)
    print(f"{method:<6s} trustworthiness (k=10): {res['trust']:.3f}")

---
## 3 — Visualize All Methods Side by Side

In [ ]:
methods = list(results.keys())
n_methods = len(methods)

fig, axes = plt.subplots(1, n_methods, figsize=(7 * n_methods, 6))
if n_methods == 1:
    axes = [axes]

for ax, method in zip(axes, methods):
    emb = results[method]["embedding"]
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap="tab10", alpha=0.7, s=15,
                    edgecolors="w", linewidth=0.2)
    ax.set_title(f"{method}\n(t = {results[method]['time']:.3f}s, "
                 f"T(10) = {results[method]['trust']:.3f})", fontsize=13)
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")

plt.suptitle("Dimensionality Reduction — Method Comparison (coloured by Facies)",
             fontsize=15, y=1.02)
plt.tight_layout(rect=[0, 0, 0.93, 1])   # lay out the panels first, leaving room on the right

# Shared colorbar
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
fig.colorbar(sc, cax=cbar_ax, label="Facies")
plt.show()

### 3.1 Stability: Two Seeds and Two Neighbourhood Settings

The protocol asks for at least two seeds and two neighbourhood settings for each non-linear
method. We use `init="random"` for t-SNE, so that the seed actually changes its starting layout.
Compare what changes between the runs (arrangement of the groups) with what does not
(trustworthiness).

In [ ]:
stability = {}
for seed in (0, 1):
    for perp in (15, 50):
        emb = TSNE(n_components=2, perplexity=perp, init="random", random_state=seed,
                   max_iter=1000).fit_transform(X_scaled)
        stability[f"t-SNE perplexity={perp}, seed={seed}"] = emb
    if UMAP_AVAILABLE:
        for nn in (15, 50):
            emb = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.1,
                            random_state=seed, n_jobs=1).fit_transform(X_scaled)
            stability[f"UMAP n_neighbors={nn}, seed={seed}"] = emb

stability_trust = {name: trustworthiness(X_scaled, emb, n_neighbors=10)
                   for name, emb in stability.items()}
for name, t in stability_trust.items():
    print(f"{name:<32s} trustworthiness (k=10): {t:.3f}")

n_cols = 4
n_rows = int(np.ceil(len(stability) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4.6 * n_rows), squeeze=False)
for ax, (name, emb) in zip(axes.ravel(), stability.items()):
    ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap="tab10", alpha=0.7, s=8)
    ax.set_title(f"{name}\nT(10) = {stability_trust[name]:.3f}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes.ravel()[len(stability):]:
    ax.axis("off")
fig.suptitle("Stability across seeds and neighbourhood settings (coloured by Facies)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 4 — Computational Time Comparison

In [ ]:
times = {m: results[m]["time"] for m in methods}

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["steelblue", "tomato", "seagreen"][:n_methods]
ax.bar(times.keys(), times.values(), color=colors, edgecolor="white", width=0.5)
ax.set_ylabel("Time (seconds)")
ax.set_title("Computational Time Comparison")

for i, (m, t) in enumerate(times.items()):
    ax.text(i, t + max(times.values()) * 0.02, f"{t:.4f}s",
            ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 5 — Facies Separation in Each Map (Silhouette Score)

The **silhouette score** of the known facies labels in each 2D map measures how separated
the facies *look* in that map. Maps can exaggerate separation (t-SNE and UMAP in particular),
so a higher map score is not evidence that the facies are separable in the logs: compare it
with the silhouette in the original 10D space (Section 6), and confirm any separation in the
logs themselves.

Note: the facies labels are used only for this check after fitting; they were never an input
of any method.

In [ ]:
sil_scores = {}
for method in methods:
    emb = results[method]["embedding"]
    sil = silhouette_score(emb, y)
    sil_scores[method] = sil
    print(f"{method:<8s} Silhouette Score: {sil:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["steelblue", "tomato", "seagreen"][:n_methods]
ax.bar(sil_scores.keys(), sil_scores.values(), color=colors, edgecolor="white", width=0.5)
ax.set_ylabel("Silhouette Score")
ax.set_title("Facies Silhouette in Each 2D Map")
ax.set_ylim(0, max(sil_scores.values()) * 1.3)

for i, (m, s) in enumerate(sil_scores.items()):
    ax.text(i, s + max(sil_scores.values()) * 0.03, f"{s:.4f}",
            ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 6 — Silhouette Score in Original High-Dimensional Space

A map score far above the 10D score means the map makes the facies look more separated than
they are in the logs.

In [ ]:
sil_orig = silhouette_score(X_scaled, y)
print(f"Silhouette Score in original {X_scaled.shape[1]}D space: {sil_orig:.4f}")
print()
for method in methods:
    diff = sil_scores[method] - sil_orig
    print(f"{method:<8s} 2D: {sil_scores[method]:.4f}  (delta = {diff:+.4f})")

---
## 7 — Summary Comparison Table

In [ ]:
summary_data = []
for method in methods:
    row = {
        "Method": method,
        "Time (s)": f"{results[method]['time']:.4f}",
        "Trustworthiness T(10)": f"{results[method]['trust']:.3f}",
        "Silhouette (2D)": f"{sil_scores[method]:.4f}",
        "Type": "Linear" if method == "PCA" else "Non-linear",
        "Transform new rows": "No" if method == "t-SNE" else "Yes",
        "Seed-dependent": "No" if method == "PCA" else "Yes (fix random_state, compare seeds)",
        "Scalability": "Excellent" if method == "PCA" else ("Good" if method == "UMAP" else "Moderate"),
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data).set_index("Method")
summary_df

---
## 8 — PCA Explained Variance Analysis

One advantage unique to PCA is the explained variance — we can quantify exactly
how much of the total variance is retained (variance, which is not necessarily the
information a task needs).

In [ ]:
pca_full = PCA(n_components=len(feature_cols), random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(feature_cols) + 1), cumvar, "o-", color="steelblue", linewidth=2)
ax.axhline(y=0.90, color="gray", linestyle="--", label="90%")
ax.axhline(y=0.95, color="gray", linestyle=":", label="95%")
ax.set_xlabel("Number of Components")
ax.set_ylabel("Cumulative Explained Variance")
ax.set_title("PCA — Cumulative Explained Variance")
ax.set_xticks(range(1, len(feature_cols) + 1))
ax.legend()
plt.tight_layout()
plt.show()

for i, cv in enumerate(cumvar):
    print(f"  {i+1} components: {cv:.4f} ({cv*100:.1f}%)")

---
## 9 — Discussion of Trade-offs

### PCA
- **Pros**: Fast, deterministic, interpretable (loadings), has inverse transform,
  quantifiable retained variance (explained variance), scales to very large datasets.
- **Cons**: Linear — cannot capture non-linear manifold structure. May not separate
  classes well when the decision boundary is non-linear.
- **Use when**: You need a fast, interpretable first pass; preprocessing for other algorithms
  (fitted on training data only, inside a pipeline, when a model is evaluated);
  the data has approximately linear structure.

### t-SNE
- **Pros**: Excellent at revealing local cluster structure; widely used for visualization.
- **Cons**: Slow for large datasets; seed-dependent (with `init="random"`); perplexity-sensitive;
  distances between clusters and cluster sizes are not meaningful;
  no transform for new points (a new well needs a refit).
- **Use when**: You want to visually explore high-dimensional data for clusters
  and the dataset is not too large (< 10k-50k points).

### UMAP
- **Pros**: Often fast on large datasets; supports transform of new data (with the same fitted
  scaler).
- **Cons**: Requires an additional package (`umap-learn`); results depend on `n_neighbors`,
  `min_dist`, the metric and the seed; claims that it preserves global structure better than
  t-SNE are debated (the initialization matters); still non-linear and not interpretable.
- **Use when**: You need non-linear dimensionality reduction at scale;
  you need to embed new unseen points.

### Practical Recommendations for Energy Industry Workflows
1. **Always start with PCA** — it is fast and gives you interpretable information
   about feature importance and variance structure.
2. **Use t-SNE or UMAP for visual exploration** — when you need to spot clusters
   or anomalies in high-dimensional well-log or production data; colour by metadata only
   after fitting.
3. **Prefer UMAP over t-SNE** when working with larger datasets or when you need
   to embed new data points into an existing projection.
4. **Combine methods** — use PCA for feature extraction and dimensionality reduction
   as preprocessing, then apply t-SNE/UMAP on the PCA output for faster visualization.
5. **Judge a map by evidence, not by how clean it looks** — trustworthiness at a stated k,
   stability across seeds and settings, and checks in the original logs.

---
## Summary

In this exercise we systematically compared three dimensionality-reduction methods:

1. **PCA** — linear, fast, interpretable, quantifiable retained variance.
2. **t-SNE** — non-linear, excellent local structure, best for small-to-medium visualization tasks.
3. **UMAP** — non-linear, neighbourhood graph, can transform new rows, often fast on large data.

We evaluated each on:
- **Visual quality** — how the facies appear in 2D (labels added after fitting)
- **Computational time** — on this machine; PCA is by far the fastest here
- **Trustworthiness and stability** — neighbourhood preservation at k = 10, across seeds and settings
- **Silhouette score** — how separated the facies look in each map, compared with the 10D value

**Key takeaway**: There is no single best method. PCA is the default starting point;
t-SNE and UMAP are powerful complements for non-linear visualization. The choice
depends on dataset size, analysis goal, and interpretability requirements, and every
map needs its settings, seeds and trustworthiness reported next to it.